
# FITS instrumental calibration: bias (zero) + dark + flat

Notebook for calibrating a single observing night using `astropy` + `numpy`.

Pipeline:

1. scan FITS headers and classify frames,
2. build a sigma-clipped median **master bias**,
3. build a **master dark-current rate** in ADU/s from bias-corrected darks,
4. build a normalized **master flat for each filter** after bias and dark correction,
5. calibrate science frames:

\[
I_{\rm cal} = \frac{I_{\rm raw} - B - t\,D}{F}
\]

where \(B\) is the master bias, \(D\) is the dark-current image in ADU/s, \(t\) is exposure time, and \(F\) is the normalized master flat.

Edit the configuration cell first so that the FITS keywords match your observatory.


In [ ]:

# If needed:
# %pip install astropy numpy matplotlib

from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import sigma_clip


## 1. Configuration

In [ ]:

# ------------------------------------------------------------------
# DIRECTORIES
# ------------------------------------------------------------------
RAW_DIR = Path("./raw")
CALIB_DIR = Path("./calib")
REDUCED_DIR = Path("./reduced")

CALIB_DIR.mkdir(parents=True, exist_ok=True)
REDUCED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# FITS STRUCTURE
# ------------------------------------------------------------------
IMAGE_HDU = 0

# Header keywords.
FRAME_TYPE_KEY = "IMAGETYP"   # often OBSTYPE in some systems
EXPTIME_KEY = "EXPTIME"
FILTER_KEY = "FILTER"
OBJECT_KEY = "OBJECT"

# Values used for different image classes.
# Comparison is case-insensitive and strips whitespace.
BIAS_TYPES = {"BIAS", "ZERO", "BIAS FRAME", "ZERO FRAME"}
DARK_TYPES = {"DARK", "DARK FRAME"}
FLAT_TYPES = {"FLAT", "FLAT FIELD", "SKY FLAT", "DOME FLAT"}
SCIENCE_TYPES = {"LIGHT", "SCIENCE", "OBJECT", "LIGHT FRAME"}

FITS_PATTERNS = ("*.fits", "*.fit", "*.fts", "*.fits.gz")

# ------------------------------------------------------------------
# COMBINATION / QUALITY PARAMETERS
# ------------------------------------------------------------------
SIGMA = 5.0
MAXITERS = 3

USE_DARK = True

# Optional trim in normal Python [y, x] convention.
# Example:
# DATA_SLICE = (slice(0, 2048), slice(0, 2048))
DATA_SLICE = None


## 2. Helper functions

In [ ]:

def normalize_type(value):
    return str(value).strip().upper()


def read_fits(path):
    # Read a 2-D FITS image as float64 and copy its header.
    with fits.open(path, memmap=False) as hdul:
        data = np.asarray(hdul[IMAGE_HDU].data, dtype=np.float64)
        header = hdul[IMAGE_HDU].header.copy()

    if data.ndim != 2:
        raise ValueError(f"{path}: expected a 2-D image, got shape {data.shape}")

    if DATA_SLICE is not None:
        data = data[DATA_SLICE]

    return data, header


def write_fits(path, data, header, overwrite=True):
    # Write a calibrated image as float32 FITS.
    path = Path(path)
    fits.PrimaryHDU(
        data=np.asarray(data, dtype=np.float32),
        header=header
    ).writeto(path, overwrite=overwrite)


def find_fits_files(directory):
    files = []
    for pattern in FITS_PATTERNS:
        files.extend(Path(directory).glob(pattern))
    return sorted(set(files))


def classify_frame(header):
    # Classify FITS as bias, dark, flat, science, or unknown.
    value = normalize_type(header.get(FRAME_TYPE_KEY, ""))

    if value in {normalize_type(x) for x in BIAS_TYPES}:
        return "bias"
    if value in {normalize_type(x) for x in DARK_TYPES}:
        return "dark"
    if value in {normalize_type(x) for x in FLAT_TYPES}:
        return "flat"
    if value in {normalize_type(x) for x in SCIENCE_TYPES}:
        return "science"

    return "unknown"


def sigma_clipped_median(stack, sigma=SIGMA, maxiters=MAXITERS):
    # Stack shape is (n_frames, ny, nx).
    clipped = sigma_clip(
        stack,
        sigma=sigma,
        maxiters=maxiters,
        axis=0,
        masked=True,
        copy=False,
    )
    return np.ma.median(clipped, axis=0).filled(np.nan)


def load_stack(paths):
    arrays = []
    shape = None

    for path in paths:
        data, _ = read_fits(path)

        if shape is None:
            shape = data.shape
        elif data.shape != shape:
            raise ValueError(
                f"Inconsistent image shape: {path} has {data.shape}, expected {shape}"
            )

        arrays.append(data)

    if not arrays:
        raise ValueError("No images supplied.")

    return np.stack(arrays, axis=0)


def exposure_from_header(header, path=None):
    try:
        t = float(header[EXPTIME_KEY])
    except Exception as exc:
        where = f" in {path}" if path is not None else ""
        raise ValueError(f"Missing/invalid {EXPTIME_KEY}{where}") from exc

    if t < 0:
        raise ValueError(f"Negative exposure time: {t}")

    return t


def filter_from_header(header):
    return str(header.get(FILTER_KEY, "UNKNOWN")).strip()


def robust_stats(data):
    finite = np.asarray(data)[np.isfinite(data)]
    return {
        "median": np.median(finite),
        "mean": np.mean(finite),
        "std": np.std(finite),
        "p01": np.percentile(finite, 1),
        "p99": np.percentile(finite, 99),
    }


## 3. Scan the observing night

In [ ]:

files = find_fits_files(RAW_DIR)
print(f"Found {len(files)} FITS files in {RAW_DIR.resolve()}")

frames = {
    "bias": [],
    "dark": [],
    "flat": [],
    "science": [],
    "unknown": [],
}

for path in files:
    try:
        with fits.open(path, memmap=True) as hdul:
            hdr = hdul[IMAGE_HDU].header
            kind = classify_frame(hdr)
            frames[kind].append(path)
    except Exception as exc:
        warnings.warn(f"Could not inspect {path}: {exc}")

for kind, paths in frames.items():
    print(f"{kind:8s}: {len(paths):4d}")

if frames["unknown"]:
    print("\nUnknown frame types:")
    for path in frames["unknown"][:20]:
        hdr = fits.getheader(path, IMAGE_HDU)
        print(
            f"  {path.name:35s} "
            f"{FRAME_TYPE_KEY}={hdr.get(FRAME_TYPE_KEY)!r} "
            f"{OBJECT_KEY}={hdr.get(OBJECT_KEY)!r}"
        )


In [ ]:

print(f"{'file':35s} {'type':9s} {'exp[s]':>10s} {'filter':>12s} object")
print("-" * 90)

for kind in ("bias", "dark", "flat", "science", "unknown"):
    for path in frames[kind]:
        hdr = fits.getheader(path, IMAGE_HDU)
        exp = hdr.get(EXPTIME_KEY, "")
        filt = hdr.get(FILTER_KEY, "")
        obj = hdr.get(OBJECT_KEY, "")
        print(
            f"{path.name:35.35s} {kind:9s} "
            f"{str(exp):>10.10s} {str(filt):>12.12s} {obj}"
        )


## 4. Master bias / zero

In [ ]:

bias_paths = frames["bias"]

if len(bias_paths) == 0:
    raise RuntimeError("No bias/zero frames found.")

bias_stack = load_stack(bias_paths)
master_bias = sigma_clipped_median(bias_stack)

bias_header = fits.getheader(bias_paths[0], IMAGE_HDU).copy()
bias_header["IMAGETYP"] = ("MASTER_BIAS", "Calibration product")
bias_header["NCOMBINE"] = (len(bias_paths), "Number of combined bias frames")
bias_header["HISTORY"] = "Sigma-clipped median combination"

master_bias_path = CALIB_DIR / "MasterBias.fits"
write_fits(master_bias_path, master_bias, bias_header)

print(master_bias_path)
print(robust_stats(master_bias))


In [ ]:

plt.figure(figsize=(8, 6))
vmin, vmax = np.nanpercentile(master_bias, [1, 99])
plt.imshow(master_bias, origin="lower", vmin=vmin, vmax=vmax)
plt.colorbar(label="ADU")
plt.title("Master bias")
plt.xlabel("x [pix]")
plt.ylabel("y [pix]")
plt.show()



## 5. Master dark-current rate

Each dark is first bias-subtracted and divided by its exposure time.  
The resulting images are in **ADU/s** and are combined with a sigma-clipped median.

This makes it possible to use darks with different exposure times, provided the dark current scales approximately linearly with exposure time.


In [ ]:

dark_paths = frames["dark"]

if USE_DARK:
    if len(dark_paths) == 0:
        raise RuntimeError("USE_DARK=True but no dark frames were found.")

    dark_rates = []
    dark_exposures = []

    for path in dark_paths:
        data, hdr = read_fits(path)
        t = exposure_from_header(hdr, path)

        if t <= 0:
            raise ValueError(f"{path}: dark exposure must be > 0 s")

        dark_rates.append((data - master_bias) / t)
        dark_exposures.append(t)

    dark_rate_stack = np.stack(dark_rates, axis=0)
    master_dark_rate = sigma_clipped_median(dark_rate_stack)

    dark_header = fits.getheader(dark_paths[0], IMAGE_HDU).copy()
    dark_header["IMAGETYP"] = ("MASTER_DARK_RATE", "Calibration product")
    dark_header["BUNIT"] = ("ADU/s", "Dark-current rate")
    dark_header["NCOMBINE"] = (len(dark_paths), "Number of combined dark frames")
    dark_header["HISTORY"] = "Bias-subtracted; divided by exposure time"
    dark_header["HISTORY"] = "Sigma-clipped median combination"

    master_dark_path = CALIB_DIR / "MasterDarkRate.fits"
    write_fits(master_dark_path, master_dark_rate, dark_header)

    print(master_dark_path)
    print("Dark exposure times [s]:", sorted(set(dark_exposures)))
    print(robust_stats(master_dark_rate))

else:
    master_dark_rate = np.zeros_like(master_bias)
    print("Dark correction disabled.")


In [ ]:

if USE_DARK:
    plt.figure(figsize=(8, 6))
    vmin, vmax = np.nanpercentile(master_dark_rate, [1, 99])
    plt.imshow(master_dark_rate, origin="lower", vmin=vmin, vmax=vmax)
    plt.colorbar(label="ADU/s")
    plt.title("Master dark-current rate")
    plt.xlabel("x [pix]")
    plt.ylabel("y [pix]")
    plt.show()



## 6. Master flats by filter

Each flat is:

1. bias-subtracted,
2. dark-corrected,
3. normalized by its own median,
4. combined with other flats from the same filter.

Normalizing individual flats before combination handles changing twilight brightness naturally.


In [ ]:

flat_paths = frames["flat"]

if len(flat_paths) == 0:
    raise RuntimeError("No flat frames found.")

flats_by_filter = {}

for path in flat_paths:
    data, hdr = read_fits(path)
    filt = filter_from_header(hdr)
    t = exposure_from_header(hdr, path)

    corrected = data - master_bias - t * master_dark_rate
    med = np.nanmedian(corrected)

    if not np.isfinite(med) or med <= 0:
        warnings.warn(f"Skipping {path.name}: invalid flat median {med}")
        continue

    normalized = corrected / med
    flats_by_filter.setdefault(filt, []).append((path, normalized))

master_flats = {}

for filt, items in sorted(flats_by_filter.items()):
    stack = np.stack([arr for _, arr in items], axis=0)
    master_flat = sigma_clipped_median(stack)

    # Normalize again after combining.
    master_flat /= np.nanmedian(master_flat)

    # Flag non-physical pixels.
    bad = (~np.isfinite(master_flat)) | (master_flat <= 0)
    master_flat[bad] = np.nan

    master_flats[filt] = master_flat

    hdr = fits.getheader(items[0][0], IMAGE_HDU).copy()
    hdr["IMAGETYP"] = ("MASTER_FLAT", "Calibration product")
    hdr["NCOMBINE"] = (len(items), "Number of combined flat frames")
    hdr["FILTER"] = (filt, "Flat-field filter")
    hdr["HISTORY"] = "Bias and dark corrected"
    hdr["HISTORY"] = "Each input normalized by median"
    hdr["HISTORY"] = "Sigma-clipped median combination"

    safe_filter = "".join(
        c if c.isalnum() or c in "-_." else "_" for c in filt
    )
    out = CALIB_DIR / f"MasterFlat_{safe_filter}.fits"
    write_fits(out, master_flat, hdr)

    print(f"{filt:12s}: {len(items):3d} flats -> {out.name}")


In [ ]:

for filt, master_flat in master_flats.items():
    plt.figure(figsize=(8, 6))
    vmin, vmax = np.nanpercentile(master_flat, [1, 99])
    plt.imshow(master_flat, origin="lower", vmin=vmin, vmax=vmax)
    plt.colorbar(label="relative response")
    plt.title(f"Master flat: {filt}")
    plt.xlabel("x [pix]")
    plt.ylabel("y [pix]")
    plt.show()


## 7. Calibrate science images

In [ ]:

science_paths = frames["science"]

if len(science_paths) == 0:
    warnings.warn("No science frames found.")

calibrated_paths = []

for path in science_paths:
    data, hdr = read_fits(path)

    t = exposure_from_header(hdr, path)
    filt = filter_from_header(hdr)

    if filt not in master_flats:
        warnings.warn(
            f"Skipping {path.name}: no master flat for filter {filt!r}"
        )
        continue

    flat = master_flats[filt]

    calibrated = data - master_bias - t * master_dark_rate
    calibrated = calibrated / flat

    hdr["CALSTAT"] = ("BDF", "Bias, dark, flat corrected")
    hdr["BIASCOR"] = (True, "Master bias subtracted")
    hdr["DARKCOR"] = (bool(USE_DARK), "Dark-current correction applied")
    hdr["FLATCOR"] = (True, "Flat-field correction applied")
    hdr["HISTORY"] = f"MasterBias: {master_bias_path.name}"

    if USE_DARK:
        hdr["HISTORY"] = f"MasterDark: {master_dark_path.name}"

    hdr["HISTORY"] = f"MasterFlat filter: {filt}"

    out = REDUCED_DIR / f"{path.stem}_cal.fits"
    write_fits(out, calibrated, hdr)
    calibrated_paths.append(out)

print(f"Calibrated {len(calibrated_paths)} / {len(science_paths)} science images.")
print(f"Output directory: {REDUCED_DIR.resolve()}")


## 8. Quick-look calibrated images

In [ ]:

for path in calibrated_paths[:5]:
    data, hdr = read_fits(path)

    vmin, vmax = np.nanpercentile(data, [5, 99.5])

    plt.figure(figsize=(8, 6))
    plt.imshow(data, origin="lower", vmin=vmin, vmax=vmax, cmap="gray")
    plt.colorbar(label="ADU")
    plt.title(
        f"{path.name}\n"
        f"{hdr.get(OBJECT_KEY, '')}  "
        f"{hdr.get(FILTER_KEY, '')}  "
        f"{hdr.get(EXPTIME_KEY, '')} s"
    )
    plt.xlabel("x [pix]")
    plt.ylabel("y [pix]")
    plt.show()



## Observatory-specific additions worth considering

This notebook deliberately does **not** assume an overscan region, gain correction, non-linearity correction, bad-pixel mask, shutter correction, or multi-extension FITS layout.

If the detector uses any of those:

- correct **overscan** before constructing/subtracting the master bias,
- apply the same **trim** to every calibration and science frame,
- if dark current is not linear in exposure time, construct master darks at matching exposure times instead of using ADU/s scaling,
- very short twilight flats can require a **shutter-pattern correction**,
- detector-specific hot/bad pixels can be turned into a **bad-pixel mask** and propagated separately.
